In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")       # First Load Data.
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)      # 2nd: Take model for Tokenizer


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)   # Convert tokenizer inside data.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)   # Make Dynamic Padding.

In [2]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [3]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

### Once we convert data in token and dynamic padding then we will set Training Parameters.

In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

#### Load the model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

#### Now make traing object.
  - 1st: Pass Model
  - 2nd: Pass Traing Arguments
  - 3rd: Take Trainging and Evaluation Datasetes
  - 4th: Maked Dynamic Padding class
  - 5th: Tell to Trainer class which tokenizer will be use.

In [7]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

#### Now claculate loss, Accuracy, F1 Score, Prediction

In [9]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


In [10]:
predictions = trainer.predict(tokenized_datasets["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(1725, 2) (1725,)


In [11]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

### Now if we want to see accuracy and f1 score after each epochs then we have to use this- Compute metrics function.

In [14]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### Here inside traing argumets we set evaluation strategy is Epoch

In [20]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch", num_train_epochs=5)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [22]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

#### Certainly! In the context of machine learning training, particularly when using a framework like Hugging Face's Trainer, a 'step' represents one iteration of the training process. Here's a breakdown:

  - Mini-Batch Processing: During training, your dataset isn't usually processed all at once. Instead, it's divided into smaller chunks called 'mini-batches' (or just 'batches').
  - One Step = One Batch: Each time the model processes one of these mini-batches, it's considered one 'step'. In this step, the model makes predictions, calculates the loss, and then updates its internal parameters (weights) based on that mini-batch's data.
  - Relationship to Epochs: An 'epoch' means the model has seen the entire training dataset once. If you have a large dataset and a small batch size, it will take many 'steps' to complete one 'epoch'.

#### Looking at your log, for example:

- 'step': 459: This means the model has processed 459 mini-batches since the beginning of training, and at this point, it performed an evaluation.

- 'step': 500: At this point, after processing 500 mini-batches, a training loss and learning rate were logged.

### In essence, 'step' is a count of how many times the model has updated its parameters using a batch of data.

### Advanced Training Features
  - Mixed Precision Training: Use fp16=True in your training arguments for faster training and reduced memory usage:

In [24]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    fp16=True,  # Enable mixed precision
)

- Gradient Accumulation: For effective larger batch sizes when GPU memory is limited:

In [25]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
)

- Learning Rate Scheduling: The Trainer uses linear decay by default, but you can customize this:

In [26]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",  # Try different schedulers
)